In [93]:
from sklearn.datasets import make_classification
import torch

# make_classification - generates FAKE data for testing/learning
X, y = make_classification(
    n_samples=10,       # 10 rows 
    n_features=2,       # 2 columns (features)
    n_informative=2,    # both features carry real signal (not noise)
    n_redundant=0,      # no useless duplicate features
    n_classes=2,        # binary classification (0 or 1)
    random_state=42    
)

In [94]:
X

array([[ 1.06833894, -0.97007347],
       [-1.14021544, -0.83879234],
       [-2.8953973 ,  1.97686236],
       [-0.72063436, -0.96059253],
       [-1.96287438, -0.99225135],
       [-0.9382051 , -0.54304815],
       [ 1.72725924, -1.18582677],
       [ 1.77736657,  1.51157598],
       [ 1.89969252,  0.83444483],
       [-0.58723065, -1.97171753]])

In [95]:
X.shape

(10, 2)

In [96]:
y

array([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [97]:
y.shape

(10,)

In [98]:
X.dtype

dtype('float64')

In [99]:
y.dtype

dtype('int64')

In [100]:
# convert numpy -> tensors with the correct dtypes
X = torch.tensor(X, dtype=torch.float32)   # float32 - required by nn.Linear
y = torch.tensor(y, dtype=torch.long)      # long (int) - required by CrossEntropyLoss

In [101]:
X.dtype

torch.float32

In [102]:
y.dtype

torch.int64

In [103]:
from torch.utils.data import Dataset, DataLoader


class CustomDataset(Dataset):

  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
      # tells PyTorch HOW MANY samples exist
      # DataLoader needs this to know when an epoch ends
      return self.features.shape[0]   

  def __getitem__(self, index):
      # tells PyTorch HOW TO FETCH one sample by its position
      # DataLoader calls this repeatedly and stacks the results into a batch
      return self.features[index], self.labels[index]   # returns a (features, label) tuple

In [104]:
dataset=CustomDataset(X,y)

In [105]:
len(dataset)

10

In [106]:
dataset[6]

(tensor([ 1.7273, -1.1858]), tensor(1))

In [107]:
# DataLoader - takes the Dataset and serves it in BATCHES automatically
dataloader = DataLoader(
    dataset,
    batch_size=2,    # 2 samples per batch (10 samples / 2 = 5 batches)
    shuffle=True     # randomize the order each epoch 
)

In [108]:
# loop through the batches - DataLoader stacks individual samples into batch tensors
for batch_features, batch_labels in dataloader:
    print(batch_features)  
    print(batch_labels)     
    print("-"*50)
    
# 10 samples / batch_size 2 = 5 batches
# NOTE: we only wrote __getitem__ to return ONE sample -
# DataLoader calls it repeatedly and STACKS the results into a batch for us

tensor([[ 1.8997,  0.8344],
        [-1.9629, -0.9923]])
tensor([1, 0])
--------------------------------------------------
tensor([[ 1.0683, -0.9701],
        [ 1.7774,  1.5116]])
tensor([1, 1])
--------------------------------------------------
tensor([[ 1.7273, -1.1858],
        [-0.9382, -0.5430]])
tensor([1, 1])
--------------------------------------------------
tensor([[-2.8954,  1.9769],
        [-1.1402, -0.8388]])
tensor([0, 0])
--------------------------------------------------
tensor([[-0.5872, -1.9717],
        [-0.7206, -0.9606]])
tensor([0, 0])
--------------------------------------------------


## Why Use Batches?

Feeding data in batches instead of all at once gives us three advantages:

**1. Memory efficiency**  
You can't fit a million samples into RAM or GPU memory at once. Batches let you train on huge datasets with limited memory.

**2. Faster convergence**   
The weights update **once per batch**, not once per epoch. With 455 samples and `batch_size=32`, that's ~14 weight updates per epoch instead of 1.

**3. The sweet spot between two extremes** 

| Approach | Batch size | Problem |
|----------|-----------|---------|
| Gradient Descent | ALL data | memory-hungry |
| SGD | 1 sample | very noisy, slow |
| **Mini-batch**  | 32, 64, 128... | **balanced — best of both** |

> `DataLoader(batch_size=32)` is exactly **Mini-batch SGD** — the approach used in practice.

## Additional DataLoader Options

Two parameters worth knowing beyond `batch_size` and `shuffle`:

**`drop_last`** — handles the uneven final batch  
If your data doesn't divide evenly, the last batch is smaller:
```
100 samples, batch_size=32 → batches of [32, 32, 32, 4]   ← last one has only 4!
```
```python
DataLoader(dataset, batch_size=32, drop_last=True)   # discards the small last batch
```
A tiny last batch can cause problems (especially with BatchNorm, which needs several samples to compute batch statistics).

**`num_workers`** — parallel loading for speed
```python
DataLoader(dataset, batch_size=32, num_workers=2)   # load batches using 2 CPU processes
```
Loads batches in parallel → faster data loading. Matters for images and large datasets; not needed for small tabular data. Default is `0` (single process).